In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray

data_dir = Path('../data')

In [2]:
metrics_dir = data_dir / 'outputs/plots/metrics/x1-y1-z1/net_cdf'

def read_plot_metrics(plot_id: str):
    metrics = xr.open_dataset(metrics_dir / f"{plot_id}.nc", decode_coords='all')
    metrics.load()
    metrics.close()
    return metrics

In [3]:
plots = gpd.read_file(data_dir / "outputs/plots/plots.geojson")
plots = plots.set_index('id')
plots

,site,plot_number,site_plot_id,geometry
id,,,,
ULO_212_P1,ULO_212,1,ULO_212_P1,"POLYGON ((460601.61 5263976.018, 460562.172 52..."
ULO_212_P2,ULO_212,2,ULO_212_P2,"POLYGON ((460579.795 5263940.548, 460537.296 5..."
ULO_212_P3,ULO_212,3,ULO_212_P3,"POLYGON ((460553.09 5263899.328, 460511.13 526..."
ULO_212_P4,ULO_212,4,ULO_212_P4,"POLYGON ((460527.43 5263862.676, 460486.481 52..."
ULO_212_P5,ULO_212,5,ULO_212_P5,"POLYGON ((460501.219 5263817.156, 460457.934 5..."
ULY_O_27_P1,ULY_O_27,1,ULY_O_27_P1,"POLYGON ((460773.837 5262517.328, 460748.749 5..."
ULY_O_27_P2,ULY_O_27,2,ULY_O_27_P2,"POLYGON ((460705.443 5262465.791, 460664.537 5..."
ULY_O_27_P3,ULY_O_27,3,ULY_O_27_P3,"POLYGON ((460749.487 5262473.667, 460773.099 5..."
ULY_O_27_P4,ULY_O_27,4,ULY_O_27_P4,"POLYGON ((460862.875 5262471.269, 460840.506 5..."


In [4]:
def create_plot_summary(row: gpd.GeoSeries) -> pd.Series:
    id = row.name
    metrics : xr.Dataset = read_plot_metrics(id)

    mean_metrics_names = [
        "point_density",
        "pulse_density",
        "scan_angle_mean",
        "chm",
        "veg_height_mean",
        "veg_height_median",
        "crr",
        "veg_height_q10",
        "veg_height_q20",
        "veg_height_q30",
        "veg_height_q40",
        "veg_height_q50",
        "veg_height_q60",
        "veg_height_q70",
        "veg_height_q80",
        "veg_height_q90",
        "veg_height_sd",
        'veg_height_cv',
        'veg_height_skew',
        'veg_height_kurt',
        'veg_height_gini',
        'canopy_cover_gt1m',
        'canopy_cover_gt1m_w',
        'fhd',
        'fhd_w',
        'vci',
        'vci_w',
        'shann_capture',
        'shann_capture_w',
        'norm_shann_capture',
        'norm_shann_capture_w'
    ]
    # Skip point and pulse density and scan angle
    sd_metric_names = mean_metrics_names[3:]
    # CV for all the ones that are in height units
    cv_metric_names = [
        "chm",
        "veg_height_mean",
        "veg_height_median",
        "veg_height_q10",
        "veg_height_q20",
        "veg_height_q30",
        "veg_height_q40",
        "veg_height_q50",
        "veg_height_q60",
        "veg_height_q70",
        "veg_height_q80",
        "veg_height_q90",
        "veg_height_sd",
    ]
    
 

    mean_metrics : pd.Series = metrics[mean_metrics_names].mean(dim=['x', 'y']).to_pandas()
    sd_metrics : pd.Series = metrics[sd_metric_names].std(dim=['x', 'y']).to_pandas()
    cv_metrics : pd.Series = (sd_metrics[cv_metric_names] / mean_metrics[cv_metric_names])
    mean_metrics = mean_metrics.add_prefix('mean__')
    sd_metrics = sd_metrics.add_prefix('sd__')
    cv_metrics = cv_metrics.add_prefix('cv__')

    # I only want max of chm
    max_metrics = pd.Series({
        "max__chm": metrics['chm'].max(dim=['x', 'y']).item()
    })

    plot_summary_metrics = pd.concat([mean_metrics, max_metrics, sd_metrics, cv_metrics])
    plot_summary_metrics.name = id

    return plot_summary_metrics

In [5]:
create_plot_summary(plots.iloc[0])

mean__point_density      1487.434800
mean__pulse_density      1487.434800
mean__scan_angle_mean       0.000000
mean__chm                  16.090184
mean__veg_height_mean       6.432255
                            ...     
cv__veg_height_q60          0.547425
cv__veg_height_q70          0.495989
cv__veg_height_q80          0.449904
cv__veg_height_q90          0.422193
cv__veg_height_sd           0.535755
Name: ULO_212_P1, Length: 73, dtype: float64

In [6]:
plot_summaries = plots.apply(create_plot_summary, axis=1)
plot_summaries['site'] = plot_summaries.index.str[0:-3]
plot_summaries

,mean__point_density,mean__pulse_density,mean__scan_angle_mean,mean__chm,mean__veg_height_mean,mean__veg_height_median,mean__crr,mean__veg_height_q10,mean__veg_height_q20,mean__veg_height_q30,...,cv__veg_height_q20,cv__veg_height_q30,cv__veg_height_q40,cv__veg_height_q50,cv__veg_height_q60,cv__veg_height_q70,cv__veg_height_q80,cv__veg_height_q90,cv__veg_height_sd,site
id,,,,,,,,,,,,,,,,,,,,,
ULO_212_P1,1487.434800,1487.434800,0.000000,16.090184,6.432255,6.234406,0.454376,2.741497,3.652962,4.544319,...,0.967918,0.796389,0.685988,0.608165,0.547425,0.495989,0.449904,0.422193,0.535755,ULO_212
ULO_212_P2,1242.220244,1242.220244,0.000000,16.525525,6.847043,6.825662,0.485700,2.876693,4.007797,5.049008,...,0.828143,0.672189,0.568080,0.495655,0.446239,0.401985,0.369146,0.358407,0.502658,ULO_212
ULO_212_P3,1408.357023,1408.357023,0.000000,13.898277,4.894582,4.733622,0.433646,1.811710,2.622987,3.327003,...,1.047994,0.847824,0.725357,0.682614,0.640759,0.593168,0.538335,0.520777,0.607151,ULO_212
ULO_212_P4,1553.529193,1553.529193,0.000000,12.860642,4.681369,4.507540,0.442068,2.305888,2.942628,3.501147,...,1.351817,1.141242,0.991045,0.881204,0.802417,0.747515,0.697412,0.651594,0.729029,ULO_212
ULO_212_P5,2579.732749,2579.732749,0.000000,23.328785,7.825799,7.384235,0.381515,3.267415,4.435634,5.515140,...,1.344986,1.125937,0.976884,0.872891,0.796737,0.729967,0.671378,0.623483,0.618763,ULO_212
ULY_O_27_P1,575.832435,331.262126,19.281403,32.785529,21.170421,23.113171,0.589534,7.874721,14.152671,18.064651,...,1.006165,0.841379,0.763064,0.709704,0.678626,0.645664,0.619125,0.591117,0.635240,ULY_O_27
ULY_O_27_P2,469.081956,301.289491,22.517012,22.321237,12.692195,13.611346,0.490163,3.427092,7.332258,9.837782,...,1.647031,1.403553,1.257936,1.155720,1.085150,1.020934,0.969195,0.912994,0.885834,ULY_O_27
ULY_O_27_P3,567.751194,330.183519,-12.805641,29.655334,17.874474,19.565078,0.536387,5.443146,10.962293,14.557228,...,1.202458,1.003899,0.899366,0.832022,0.784243,0.748370,0.705770,0.663977,0.657351,ULY_O_27
ULY_O_27_P4,477.223022,285.855087,-7.180584,30.441850,19.007135,20.892672,0.568232,6.183010,11.928011,15.565061,...,1.107755,0.921526,0.805745,0.736341,0.687834,0.644472,0.603163,0.562243,0.588859,ULY_O_27


In [7]:
csv_dir = Path('../csvs')
plot_summaries.to_csv(csv_dir / 'plot_all_metrics.csv')

In [8]:
site_summaries = plot_summaries.reset_index().drop(columns='id').groupby('site').mean(numeric_only=True)
site_summaries = site_summaries[~site_summaries.index.str.startswith('AGG')]
# site_summaries
site_summaries.to_csv(csv_dir / "site_all_metrics.csv")